In [20]:
import sys
sys.path.append('/home/magesh/TrandingProjects/Project/backend/dolphin/TradingStradegy')

In [21]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from ta import add_all_ta_features
import ta
from advanced_ta import LorentzianClassification
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from mt4stradegies import binary_arrows,extreme_binary,extreme_spike,iim_arrows,super_arrows,super_signals,super_signals_v3,tm_indicator

In [22]:
prediction = {
    'NEUTRAL': 0,
    'BUY': 1,
    'SELL': 2
}

In [23]:
from datetime import timedelta
def indicatorCalculations(data_1 : pd.DataFrame):
    data = data_1.copy()
    data['UTC'] = pd.to_datetime(data['datetime']) + timedelta(hours=5)
    data['GMT'] = data['UTC'] + timedelta(hours=2)
    b_arrow = binary_arrows.BinaryArrowSignalPredictor(data.iloc[::-1].reset_index(drop=True)).run()
    # extreme_b = extreme_binary.ExtremeBinarySignalPredictor(data).run() # works only in 15 mins chart
    # spike_e = extreme_spike.ExtremeSpikeSignalPredictor(b_arrow.iloc[::-1]).run()
    # arrow_imm = iim_arrows.IINWMARROWSSignalPredictor(spike_e).run()
    # arrow_super = super_arrows.SuperArrowSignalPredictor(spike_e).run()
    super_sig = super_signals.SuperSignalPredictor(b_arrow).run()
    super_sig_v3 = super_signals_v3.SuperV3SignalPredictor(super_sig).run()
    new_data = tm_indicator.TMIndicator(super_sig_v3).run()
    # # Check that all predictors return DataFrames with the same index
    # data_frames = [data.iloc[:, :8], b_arrow, spike_e, arrow_imm, arrow_super, super_sig, super_sig_v3, tmind]
    # # Ensure all DataFrames have the same index
    # for df in data_frames:
    #     if not df.index.equals(data.index):
    #         df.index = data.index
    # new_data = pd.concat(data_frames, axis=1)
    new_data['next_close'] = new_data['close'].shift(-3)
    # new_data = pd.concat([data.iloc[:,:8],b_arrow,spike_e,arrow_imm,arrow_super,super_sig,super_sig_v3,tmind])
    return new_data

In [24]:
def calculate_1(pddata: pd.DataFrame, predict=True):
    # Add indicators to new DataFrame
    pd2 = indicatorCalculations(pddata)
    # pd2['ExtremeSignal'] = np.where(
    # (pd2['line1'] == 1) | (pd2['line5'] == 1), 1,
    # np.where((pd2['line2'] == 2) | (pd2['line5'] == 2), 2, 0)
    # )
    signals = ['datetime','symbol','open','high','low','close','next_close','volume',
                'BinaryArrow',
                'SuperSignalV3','TMSignal','UTC','GMT']
    predit_signals = [
    'BinaryArrow',
     'SuperSignalV3'
    ]
    
    pd2[predit_signals] = pd2[predit_signals].shift(1)
    all_signals_zero = (pd2[predit_signals] == 0).all(axis=1)
    pd2 = pd2[signals]
    pd2['Prediction'] = np.where(
                        all_signals_zero, 0,
                        np.where(pd2['open'] < pd2['next_close'], 1,
                                np.where(pd2['open'] > pd2['next_close'], 2, 0))
                    ).astype('int32')
    
    # pd2 = pd2[(pd2[signals] != 0).any(axis=1)]
    
    
    # pd2.dropna(inplace=True)

    return pd2

In [25]:
def process_files(file_paths):
    pd_data = []
    for file_path in file_paths:
        df = pd.read_csv(file_path)
        cal = calculate_1(df)
        pd_data.append(cal)
    return pd.concat(pd_data)

first_list = ['EURUSD', 'EURCAD', 'EURJPY', 'EURGBP', 'EURAUD'] # 
sc_list = ['EURUSD', 'EURCAD', 'EURJPY', 'EURGBP', 'USDCAD', 'USDJPY']
th_list = ['EURAUD', 'EURUSD', 'EURCAD', 'EURJPY', 'EURGBP', 'USDCAD', 'USDJPY']

file_paths = []
# file_paths = ['/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/five_mins/EURJPY_5_Min_testing_new.csv']
for curr in first_list:
    file_paths.append(f"/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/five_mins/{curr}_5_Min.csv")

for curr in sc_list:
    file_paths.append(f'/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/five_mins/{curr}_5_Min_1.csv')

for curr in th_list:
    file_paths.append(f'/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/five_mins/{curr}_5_Min_2.csv')

for curr in th_list:
    file_paths.append(f'/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/five_mins/{curr}_5_Min_3.csv')
for curr in th_list:
    file_paths.append(f'/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/five_mins/{curr}_5_Min_4.csv')

data = process_files(file_paths)

In [26]:
data.to_csv('/tmp/fulldata.csv')

In [27]:
data.dropna(inplace=True)
data.reset_index(drop=True,inplace=True)
print(data.iloc[:,8:].head())
print(data.shape)

   BinaryArrow  SuperSignalV3  TMSignal                 UTC  \
0          0.0            0.0         0 2024-03-27 14:05:00   
1          0.0            0.0         0 2024-03-27 14:00:00   
2          0.0            0.0         0 2024-03-27 13:55:00   
3          0.0            0.0         0 2024-03-27 13:50:00   
4          0.0            0.0         0 2024-03-27 13:45:00   

                  GMT  Prediction  
0 2024-03-27 16:05:00           0  
1 2024-03-27 16:00:00           0  
2 2024-03-27 15:55:00           0  
3 2024-03-27 15:50:00           0  
4 2024-03-27 15:45:00           0  
(212798, 14)


In [28]:
le = LabelEncoder()
le.fit_transform(data['Prediction'])
print(le.classes_)

[0 1 2]


In [29]:
print(data['Prediction'].value_counts())
X = data.iloc[:,8:-3]
y = data.iloc[:, -1]
print(data.columns)
print(X.columns)
print(X.count())
# print(y.head())
X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.35,train_size=0.65, shuffle=False)

Prediction
0    189317
2     11894
1     11587
Name: count, dtype: int64
Index(['datetime', 'symbol', 'open', 'high', 'low', 'close', 'next_close',
       'volume', 'BinaryArrow', 'SuperSignalV3', 'TMSignal', 'UTC', 'GMT',
       'Prediction'],
      dtype='object')
Index(['BinaryArrow', 'SuperSignalV3', 'TMSignal'], dtype='object')
BinaryArrow      212798
SuperSignalV3    212798
TMSignal         212798
dtype: int64


In [30]:
data.to_csv('/tmp/results.csv')

In [31]:
# Initialize XGBoost classifier
xgb_model = XGBClassifier(booster="gbtree",max_depth=14,min_child_weight = 2)
# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
preds = xgb_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by XGBoost Classifier\
: {accuracy_score(y_train, xgb_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by XGBoost Classifier\
: {accuracy_score(y_test, preds)*100}")


Accuracy on train data by XGBoost Classifier: 96.32585780592548
Accuracy on test data by XGBoost Classifier: 97.00590762620837


In [32]:

rf_model = RandomForestClassifier()
# Train the model
rf_model.fit(X_train, y_train)

# Make predictions on the test set  
preds = rf_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by RandomForest Classifier\
: {accuracy_score(y_train, rf_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by RandomForest Classifier\
: {accuracy_score(y_test, preds)*100}")

Accuracy on train data by RandomForest Classifier: 96.32585780592548
Accuracy on test data by RandomForest Classifier: 97.00725026852847


In [33]:
final_rf_model = RandomForestClassifier()
final_rf_model.fit(X, y)

RandomForestClassifier()

In [34]:
final_xgb_model = XGBClassifier(booster="gbtree",max_depth=14,min_child_weight = 2)
final_xgb_model.fit(X, y)

XGBClassifier(base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=14, max_leaves=None,
              min_child_weight=2, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [35]:
df = pd.read_csv('/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing_new.csv')
cal = calculate_1(df)
test_data = cal.iloc[:,8:-3]
cal['rf_predict'] = rf_model.predict(test_data)
cal['xgb_predict'] = xgb_model.predict(test_data)
cal.to_csv('/tmp/newoneresultsusd.csv')

In [37]:
import pickle
xgb_final_model = pickle.dump(final_xgb_model, open('/home/magesh/TrandingProjects/Project/backend/dolphin/common/ml_model/xgbclassifier_new_5.sav','wb'))
rf_final_model = pickle.dump(final_rf_model, open('/home/magesh/TrandingProjects/Project/backend/dolphin/common/ml_model/rfclassifier_new_5.sav','wb'))